# Pegangan UTS Data Mining II — Text

**Cara pakai: import → konfigurasi analisis → sisakan SATU chunk loading → cek data → analisis.**

Hapus semua chunk loading yang tidak dipakai, termasuk judul dan cell kodenya. Setelah
itu baru Run All. Tidak ada pemilih `SOURCE`. Semua alternatif menghasilkan `df`,
`TEXT_COLUMN`, dan `LABEL_COLUMN`, sehingga preprocessing setelahnya tetap sama.

| Chunk | Input |
|---|---|
| A | CSV lokal |
| B | Excel lokal |
| C | Dataset Kaggle berisi CSV/Excel |

Lokasi `/content/...` adalah contoh Colab. Ganti sesuai lokasi data di komputer.
Notebook belum dijalankan pada dataset. Default stopword Sastrawi tidak memerlukan corpus
NLTK; siapkan package sebelum ujian jika internet tidak tersedia.


## 1. Package dan import


In [ ]:
# Jalankan hanya jika diperlukan:
# %pip install numpy pandas matplotlib scikit-learn Sastrawi nltk openpyxl wordcloud


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score


## 2. Konfigurasi preprocessing bersama
Path, kolom teks, dan kolom label berada di masing-masing chunk loading.


In [ ]:
STOPWORD_SOURCE = "sastrawi"  # sastrawi / nltk / none
LANGUAGE = "indonesian"
CUSTOM_STOPWORDS = []
KEEP_WORDS = []  # misalnya ["tidak", "bukan"] bila negasi perlu dipertahankan
USE_STEMMING = False
REMOVE_NUMBERS = True
DROP_DUPLICATES = False
MAX_ITEMS = None
RANDOM_STATE = 42


In [ ]:
MAX_FEATURES = 5000  # None bila ingin seluruh kosakata
MIN_DF = 1
MAX_DF = 1.0
NGRAM_RANGE = (1, 1)  # (1, 2) untuk unigram + bigram

K_FIXED = None
K_CANDIDATES = [2, 3, 4, 5, 6, 7, 8, 10]
SILHOUETTE_SAMPLE = 1000
USE_MINIBATCH = False
TOP_N_WORDS = 10


# PILIH SATU CHUNK LOADING — hapus alternatif lainnya


## CHUNK A — CSV lokal

Ganti path dan nama kolom. Jika seluruh data terbaca sebagai satu kolom, periksa separator. `LABEL_COLUMN=None` jika label tidak tersedia.


In [ ]:
DATA_PATH = Path("/content/text.csv")
TEXT_COLUMN = "content"
LABEL_COLUMN = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"

df = pd.read_csv(DATA_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
print("Shape:", df.shape)
print("Kolom:", list(df.columns))
display(df.head())


## CHUNK B — Excel lokal

Ganti path, sheet, dan nama kolom. Sheet dapat berupa nomor (0 untuk pertama) atau nama. `.xls` lama membutuhkan xlrd.


In [ ]:
DATA_PATH = Path("/content/text.xlsx")
TEXT_COLUMN = "content"
LABEL_COLUMN = None
SHEET_NAME = 0

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
print("Shape:", df.shape)
print("Kolom:", list(df.columns))
display(df.head())


## CHUNK C — CSV/Excel dari Kaggle

Ganti slug dataset dan `FILE_IN_DATASET` sesuai isi folder yang dicetak. Slug diambil dari URL `kaggle.com/datasets/pemilik/nama-dataset`. File CSV/Excel dibaca di dalam chunk ini.


In [ ]:
# %pip install kagglehub
import kagglehub

KAGGLE_DATASET = "pemilik/nama-dataset"  # wajib diganti sesuai dataset soal
FILE_IN_DATASET = "data.csv"  # contoh: "folder/BSI-reviews.csv"
TEXT_COLUMN = "content"
LABEL_COLUMN = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"
SHEET_NAME = 0

dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Lokasi dataset:", dataset_path)
print("Contoh isi dataset:", [str(p.relative_to(dataset_path)) for p in dataset_path.rglob("*") if p.is_file()][:20])


In [ ]:
DATA_PATH = dataset_path / FILE_IN_DATASET
if DATA_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
else:
    df = pd.read_csv(DATA_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
print("Shape:", df.shape)
print("Kolom:", list(df.columns))
display(df.head())


# CEK DATA DAN ANALISIS BERSAMA — semua input lanjut dari sini

Pertahankan seluruh cell setelah judul ini. Pastikan nama kolom yang diisi pada chunk
terpilih cocok dengan kolom tabel. Satu teks menjadi satu unit clustering.


In [ ]:
df.info()
display(df.isna().sum().rename("missing").to_frame())


In [ ]:
if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Periksa TEXT_COLUMN. Kolom tersedia: {list(df.columns)}")
if LABEL_COLUMN is not None and LABEL_COLUMN not in df.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")

df_data = pd.DataFrame({"row_id": np.arange(len(df)), "text_raw": df[TEXT_COLUMN]})
df_data["label"] = df[LABEL_COLUMN] if LABEL_COLUMN is not None else None
print("Missing pada teks:", df_data["text_raw"].isna().sum())
df_data = df_data.dropna(subset=["text_raw"]).copy()
df_data["text_raw"] = df_data["text_raw"].astype(str).str.strip()
empty_count = df_data["text_raw"].eq("").sum()
df_data = df_data.loc[df_data["text_raw"].ne("")].copy()
print("Teks kosong dibuang:", empty_count)
print("Duplikat teks:", df_data["text_raw"].duplicated().sum())
if DROP_DUPLICATES:
    df_data = df_data.drop_duplicates(subset=["text_raw"])
total_rows = len(df_data)
if MAX_ITEMS is not None and len(df_data) > MAX_ITEMS:
    df_data = df_data.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)
df_data = df_data.reset_index(drop=True)
if df_data.empty:
    raise ValueError("Tidak ada teks valid.")
print("Teks dipakai:", len(df_data), "dari", total_rows)


## 4. EDA singkat
`text_length` menghitung karakter, bukan jumlah token. Wordcloud opsional tersedia di akhir notebook.


In [ ]:
df_data["text_length"] = df_data["text_raw"].str.len()
display(df_data["text_length"].describe().to_frame())

plt.figure(figsize=(8, 4))
plt.hist(df_data["text_length"], bins=30)
plt.xlabel("Jumlah karakter")
plt.ylabel("Jumlah dokumen")
plt.title("Distribusi panjang teks")
plt.tight_layout()
plt.show()


## 5. Cleaning

Huruf kecil → buang URL/mention → hapus angka jika dipilih → ganti tanda baca dengan spasi
→ rapikan spasi. Kata hashtag tetap dipertahankan tanpa simbol `#`.
Penggantian dengan spasi menghindari penggabungan kata di kedua sisi tanda baca.
Jika angka/emoji/tanda baca penting bagi soal, sesuaikan aturan sebelum preprocessing.


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    if REMOVE_NUMBERS:
        text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]|_", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df_data["text_clean"] = df_data["text_raw"].apply(clean_text)
display(df_data[["text_raw", "text_clean"]].head())


## 6. Tokenisasi

Setelah tanda baca dibersihkan, `.split()` memisahkan teks berdasarkan spasi tanpa download
tokenizer. Ini cukup sebagai baseline bag-of-words. Jika soal meminta `word_tokenize`,
gunakan alternatif pada catatan akhir dan siapkan corpus NLTK sebelumnya.


In [ ]:
df_data["tokens"] = df_data["text_clean"].str.split()
display(df_data[["text_clean", "tokens"]].head())


## 7. Stopword — pilih Sastrawi, NLTK, atau tanpa stopword

Daftar stopword NLTK dan Sastrawi berbeda. Sastrawi pada tahap ini hanya menyediakan
daftar stopword, belum melakukan stemming. Negasi dapat penting, misalnya untuk ulasan;
masukkan kata yang ingin dipertahankan ke `KEEP_WORDS`.


In [ ]:
if STOPWORD_SOURCE == "sastrawi":
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stop_words = set(StopWordRemoverFactory().get_stop_words())
elif STOPWORD_SOURCE == "nltk":
    import nltk
    from nltk.corpus import stopwords
    try:
        stop_words = set(stopwords.words(LANGUAGE))
    except LookupError:
        raise LookupError("Corpus stopwords belum tersedia. Jalankan nltk.download('stopwords') saat ada internet.")
elif STOPWORD_SOURCE == "none":
    stop_words = set()
else:
    raise ValueError("STOPWORD_SOURCE harus sastrawi, nltk, atau none.")

stop_words.update(word.lower() for word in CUSTOM_STOPWORDS)
stop_words.difference_update(word.lower() for word in KEEP_WORDS)
df_data["tokens_no_stopwords"] = df_data["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words])
print("Jumlah stopword:", len(stop_words))
display(df_data[["tokens", "tokens_no_stopwords"]].head())


## 8. Stemming opsional
Stemming Sastrawi mengubah kata berimbuhan ke bentuk dasar. Langkah ini dapat lambat dan tidak selalu meningkatkan kualitas cluster.


In [ ]:
df_data["text_processed"] = df_data["tokens_no_stopwords"].apply(" ".join)

if USE_STEMMING:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    df_data["text_processed"] = df_data["text_processed"].apply(stemmer.stem)

empty_count = df_data["text_processed"].str.strip().eq("").sum()
print("Teks kosong setelah preprocessing dibuang:", empty_count)
df_data = df_data.loc[df_data["text_processed"].str.strip().ne("")].reset_index(drop=True)
if len(df_data) < 3:
    raise ValueError("Kurang dari 3 teks setelah preprocessing. Periksa aturan cleaning/stopword.")
display(df_data[["row_id", "text_raw", "text_processed"]].head())


## 9. TF-IDF

Baris = dokumen, kolom = kata/ngram, nilai = bobot TF-IDF. Matriks tetap sparse agar hemat
memori. TF-IDF default sudah memakai normalisasi L2; tidak ditambah StandardScaler.
Token satu huruf diizinkan supaya teks pendek tidak otomatis hilang seluruhnya.


In [ ]:
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
                           ngram_range=NGRAM_RANGE, token_pattern=r"(?u)\b\w+\b")
try:
    X_tfidf = vectorizer.fit_transform(df_data["text_processed"])
except ValueError as error:
    raise ValueError("TF-IDF gagal. Periksa teks kosong, MIN_DF/MAX_DF, atau kosakata yang habis.") from error

nonzero = np.asarray(X_tfidf.getnnz(axis=1)).ravel() > 0
print("Dokumen tanpa fitur TF-IDF dibuang:", (~nonzero).sum())
df_data = df_data.loc[nonzero].reset_index(drop=True)
X_tfidf = X_tfidf[nonzero]
print("Shape TF-IDF:", X_tfidf.shape)
print("Contoh kosakata:", vectorizer.get_feature_names_out()[:20])


## Clustering — pilih k atau ikuti k dari soal

Ubah `K_FIXED` di konfigurasi jika soal menentukan jumlah cluster. Jika `None`,
bandingkan `K_CANDIDATES` dan pilih silhouette tertinggi pada kandidat yang dicoba.
Ini pemilihan eksploratif pada data yang sama, bukan skor generalisasi test.

`X_model` adalah fitur yang dipakai K-Means. Silhouette dihitung pada ruang fitur ini,
bukan pada koordinat grafik dua dimensi. Untuk data besar, skor menggunakan maksimal
`SILHOUETTE_SAMPLE` baris acak yang sama pada setiap k. Skor sampel merupakan perkiraan.


In [ ]:
X_model = X_tfidf

n_samples = X_model.shape[0]
if n_samples < 3:
    raise ValueError("Perlu minimal 3 data valid untuk alur clustering dan silhouette ini.")

k_values = [K_FIXED] if K_FIXED is not None else K_CANDIDATES
k_values = sorted(set(k for k in k_values if 2 <= k < n_samples))
if not k_values:
    raise ValueError("K harus minimal 2 dan lebih kecil dari jumlah data. Periksa konfigurasi.")

rng = np.random.default_rng(RANDOM_STATE)
eval_size = n_samples if SILHOUETTE_SAMPLE is None else min(SILHOUETTE_SAMPLE, n_samples)
if eval_size < 3:
    raise ValueError("SILHOUETTE_SAMPLE harus minimal 3, atau None.")
eval_index = rng.choice(n_samples, size=eval_size, replace=False)
X_eval = X_model[eval_index]
print("Data modeling:", X_model.shape, "| Data evaluasi silhouette:", eval_size)


In [ ]:
results = []
models = {}

for k in k_values:
    if USE_MINIBATCH:
        model = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                                n_init=10, batch_size=1024)
    else:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)

    labels = model.fit_predict(X_model)
    eval_labels = labels[eval_index]
    score = np.nan

    if 2 <= len(np.unique(eval_labels)) < len(eval_labels):
        score = silhouette_score(X_eval, eval_labels, metric="euclidean")

    models[k] = model
    results.append({"k": k, "cluster_terbentuk": len(np.unique(labels)),
                    "silhouette": score, "inertia": model.inertia_})
    print(f"k={k} | silhouette={score:.4f} | inertia={model.inertia_:.2f}")

df_scores = pd.DataFrame(results)
display(df_scores)


`NaN` pada silhouette berarti sampel evaluasi tidak memenuhi syarat jumlah cluster.
Jika cluster sangat kecil terlewat oleh sampling, naikkan `SILHOUETTE_SAMPLE` atau
gunakan `None`. Data identik dapat menghasilkan lebih sedikit cluster daripada k.


In [ ]:
if K_FIXED is not None:
    best_k = K_FIXED
else:
    valid_scores = df_scores.dropna(subset=["silhouette"])
    if valid_scores.empty:
        raise ValueError("Tidak ada silhouette valid. Periksa data identik atau ukuran sampel evaluasi.")
    best_k = int(valid_scores.loc[valid_scores["silhouette"].idxmax(), "k"])

kmeans = models[best_k]
df_result = df_data.copy()
df_result["cluster"] = kmeans.predict(X_model)
print("K yang dipakai:", best_k)
display(df_result.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df_scores["k"], df_scores["silhouette"], marker="o")
axes[0].set_title("Silhouette per k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette")

axes[1].plot(df_scores["k"], df_scores["inertia"], marker="o")
axes[1].set_title("Elbow / inertia")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Inertia")
plt.tight_layout()
plt.show()


## Distribusi dan evaluasi tambahan

Silhouette mendekati 1 berarti pemisahan lebih baik; sekitar 0 berarti cluster
bertumpang tindih; nilai negatif menunjukkan sebagian data lebih dekat ke cluster lain.
Inertia biasanya turun ketika k naik; cari perubahan kemiringan untuk membantu membaca elbow.

ARI dan NMI hanya dihitung jika ada label acuan yang bermakna. Nomor cluster tidak
otomatis sama dengan label kelas. Jangan langsung memakai accuracy pada nomor cluster.


In [ ]:
cluster_count = df_result["cluster"].value_counts().sort_index()
display(cluster_count.rename("jumlah").to_frame())

cluster_count.plot(kind="bar", figsize=(7, 4))
plt.xlabel("Cluster")
plt.ylabel("Jumlah data")
plt.title("Distribusi cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
label_mask = df_result["label"].notna()
if label_mask.sum() >= 2 and df_result.loc[label_mask, "label"].nunique() >= 2:
    y_true = df_result.loc[label_mask, "label"].astype(str)
    y_cluster = df_result.loc[label_mask, "cluster"]

    print("Jumlah data dengan label acuan:", len(y_true))
    print("ARI:", round(adjusted_rand_score(y_true, y_cluster), 4))
    print("NMI:", round(normalized_mutual_info_score(y_true, y_cluster), 4))
    display(pd.crosstab(y_true, y_cluster, rownames=["Label acuan"], colnames=["Cluster"]))
else:
    print("ARI/NMI dilewati: label acuan tidak tersedia atau hanya ada satu kelas.")


## Visualisasi TruncatedSVD

TruncatedSVD bekerja langsung pada matriks sparse TF-IDF. SVD tidak melakukan centering
seperti PCA, sehingga sumbu diberi nama komponen SVD. K-Means tetap memakai TF-IDF asal.
Jika kosakata hanya satu fitur, tampilkan fitur tersebut dengan sumbu kedua nol.


In [ ]:
if X_tfidf.shape[1] >= 2:
    svd = TruncatedSVD(n_components=min(2, X_tfidf.shape[1]), random_state=RANDOM_STATE)
    X_svd = svd.fit_transform(X_tfidf)
    print("Total explained variance:", round(svd.explained_variance_ratio_.sum(), 4))
else:
    X_svd = X_tfidf.toarray()
    print("Hanya satu fitur; reduksi SVD dilewati.")
plot_y = X_svd[:, 1] if X_svd.shape[1] > 1 else np.zeros(len(X_svd))

plt.figure(figsize=(8, 6))
for cluster in sorted(df_result["cluster"].unique()):
    mask = df_result["cluster"].to_numpy() == cluster
    plt.scatter(X_svd[mask, 0], plot_y[mask], label=f"Cluster {cluster}", alpha=0.6, s=20)
plt.xlabel("Komponen SVD 1")
plt.ylabel("Komponen SVD 2" if X_svd.shape[1] > 1 else "Tidak ada komponen kedua")
plt.title("Visualisasi cluster teks")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Interpretasi — kata utama tiap cluster

Kata diurutkan berdasarkan bobot pada centroid, yaitu rata-rata TF-IDF anggota cluster
untuk K-Means biasa. MiniBatch memakai centroid hasil pembaruan bertahap.
Kata berbobot tinggi membantu memberi nama topik; periksa juga contoh teksnya.


In [ ]:
terms = vectorizer.get_feature_names_out()
top_word_rows = []
for cluster in sorted(df_result["cluster"].unique()):
    weights = kmeans.cluster_centers_[cluster]
    top_indices = weights.argsort()[::-1][:min(TOP_N_WORDS, len(terms))]
    top_indices = top_indices[weights[top_indices] > 0]
    words = terms[top_indices]
    print(f"Cluster {cluster}:", ", ".join(words))
    for word, weight in zip(words, weights[top_indices]):
        top_word_rows.append({"cluster": cluster, "word": word, "weight": weight})
    plt.figure(figsize=(7, 3.5))
    plt.barh(words[::-1], weights[top_indices][::-1])
    plt.xlabel("Bobot centroid TF-IDF")
    plt.title(f"Kata utama cluster {cluster}")
    plt.tight_layout()
    plt.show()
df_top_words = pd.DataFrame(top_word_rows)


## Interpretasi — contoh dokumen tiap cluster
Baca contoh dokumen sebelum menyimpulkan nama topik. Clustering topik tidak otomatis menghasilkan label sentimen.


In [ ]:
for cluster in sorted(df_result["cluster"].unique()):
    print("Cluster", cluster)
    display(df_result.loc[df_result["cluster"] == cluster, ["row_id", "text_raw", "label"]].head(5))


## Wordcloud opsional
Hanya untuk eksplorasi kata yang sering muncul; wordcloud tidak menjadi fitur atau bukti kualitas clustering.


In [ ]:
SHOW_WORDCLOUD = False

if SHOW_WORDCLOUD:
    from wordcloud import WordCloud
    all_text = " ".join(df_data["text_processed"])
    cloud = WordCloud(width=800, height=400, background_color="white", collocations=False).generate(all_text)
    plt.figure(figsize=(10, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()


## Simpan hasil (opsional)

Ubah `SAVE_RESULTS = True` jika ingin membuat CSV. Semua baris hasil mengikuti urutan
data yang benar-benar berhasil diproses. File gagal tersedia terpisah pada `df_failed`
jika notebook ini membaca kumpulan file.


In [ ]:
SAVE_RESULTS = False
OUTPUT_DIR = Path("hasil_uts")

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_result.to_csv(OUTPUT_DIR / "text_hasil.csv", index=False)
    df_scores.to_csv(OUTPUT_DIR / "text_skor_k.csv", index=False)
    print("Hasil disimpan di:", OUTPUT_DIR.resolve())


## Catatan saat UTS

- Perbandingan NLTK/Sastrawi: ubah `STOPWORD_SOURCE` dan rerun dari konfigurasi, dengan data,
  random seed, kandidat k, dan aturan preprocessing lain tetap sama. Daftar stopword berbeda;
  jangan menyebut Sastrawi sudah melakukan stemming bila `USE_STEMMING=False`.
- Jika soal meminta tokenizer NLTK, siapkan `nltk.download('punkt_tab')`, lalu ganti cell tokenisasi
  dengan `from nltk.tokenize import word_tokenize` dan
  `df_data["tokens"] = df_data["text_clean"].apply(word_tokenize)`.
- Untuk train/test, fit TF-IDF dan model hanya pada train. Pada test, lakukan cleaning yang sama,
  `vectorizer.transform(teks_test)`, lalu `kmeans.predict(...)`. Jangan fit ulang vectorizer pada test.
- Bila ingin melihat sebagian matriks TF-IDF, ambil potongan kecil dulu:
  `pd.DataFrame(X_tfidf[:5, :10].toarray(), columns=terms[:10])`.
  Hindari `.toarray()` pada seluruh matriks besar.

- Jika soal menetapkan k, ubah `K_FIXED`, lalu jalankan kembali bagian clustering dan hasil setelahnya.
- Jika konfigurasi input atau preprocessing berubah, jalankan ulang dari konfigurasi sampai akhir.
- `MAX_ITEMS = None` berarti memakai seluruh data; angka tertentu berarti sampel acak yang dilaporkan.
- Untuk data besar, coba `USE_MINIBATCH = True` bila soal mengizinkan MiniBatch K-Means;
  hasilnya dapat berbeda dari K-Means biasa. Silhouette sampling hanya mempercepat evaluasi,
  sedangkan `MAX_ITEMS` atau MiniBatch membantu mengurangi beban modeling.
- Sesuaikan metode dengan instruksi soal. Template ini berfokus pada clustering seperti notebook contoh.
- Hasil k terbaik hanya terbaik di antara kandidat dan representasi yang dicoba; tidak menjamin cluster
  tersebut paling bermakna. Jelaskan juga isi cluster dan keterbatasan fitur.


Referensi API: [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
[TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html),
[silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

Kaggle: [dokumentasi resmi kagglehub](https://github.com/Kaggle/kagglehub).
